In [2]:
#!/usr/bin/env python3
"""
Reproducing time-dependent CL spectra S_CL(omega, t)
for a 4-level V-type emitter with electron-induced interference.

Implements:
S_CL(ω,t) = Re ∫_0^t dt2 ∫_0^{t-t2} dτ e^{-Γ(t-t2)} e^{(Γ/2 - i ω)τ}
            Σ_{i,j=1}^3 γ_ij <A_{i0}(t2+τ) A_{0j}(t2)>

Master equation Liouvillian includes:
- Hamiltonian H0 = Σ_{n=1}^3 ω_{n0} |n><n|
- Radiative decay with (γ + r) width
- Incoherent pump r
- Interference cross terms r_ij = p sqrt(r_i r_j)
- Nonradiative decay 3->2, 3->1, 2->1 at rate γ_nr

Outputs:
- figure2_repro.png (2x2 panels, 3D waterfall)
- NPZ/CSV data per panel and combined NPZ

Units:
- We set γ = 1 (so time is in γ^{-1} and frequencies in γ).
"""

from __future__ import annotations

import os
import json
from dataclasses import dataclass, asdict
from typing import Dict, Tuple, List

import numpy as np
from numpy.typing import NDArray
from scipy.linalg import expm

import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

try:
    import pandas as pd
    HAVE_PANDAS = True
except Exception:
    HAVE_PANDAS = False

In [3]:
# ----------------------------
# Linear-algebra utilities
# ----------------------------

def ketbra(d: int, m: int, n: int) -> NDArray[np.complex128]:
    M = np.zeros((d, d), dtype=np.complex128)
    M[m, n] = 1.0
    return M

def vec(X: NDArray[np.complex128]) -> NDArray[np.complex128]:
    # column-stacking vectorization
    return X.reshape((-1,), order="F")

def unvec(x: NDArray[np.complex128], d: int) -> NDArray[np.complex128]:
    return x.reshape((d, d), order="F")

def super_left(A: NDArray[np.complex128]) -> NDArray[np.complex128]:
    d = A.shape[0]
    return np.kron(np.eye(d, dtype=np.complex128), A)

def super_right(B: NDArray[np.complex128]) -> NDArray[np.complex128]:
    d = B.shape[0]
    return np.kron(B.T, np.eye(d, dtype=np.complex128))

def super_A_X_B(A: NDArray[np.complex128], B: NDArray[np.complex128]) -> NDArray[np.complex128]:
    # vec(A X B) = (B^T ⊗ A) vec(X)
    return np.kron(B.T, A)

def commutator_super(H: NDArray[np.complex128]) -> NDArray[np.complex128]:
    # -i[H,·] with ħ=1
    d = H.shape[0]
    I = np.eye(d, dtype=np.complex128)
    return -1j * (np.kron(I, H) - np.kron(H.T, I))

def lindblad_super(C: NDArray[np.complex128], rate: float) -> NDArray[np.complex128]:
    # rate * (C ρ C† - 1/2 {C†C, ρ})
    d = C.shape[0]
    if rate == 0.0:
        return np.zeros((d*d, d*d), dtype=np.complex128)
    CdC = C.conj().T @ C
    return rate * (super_A_X_B(C, C.conj().T) - 0.5 * (super_left(CdC) + super_right(CdC)))

In [4]:
# ----------------------------
# Model parameters and Liouvillian
# ----------------------------

@dataclass
class Params:
    # radiative decay (all equal in the paper's numerics)
    gamma: float = 1.0

    # incoherent excitation rate r (assumed equal for i=1,2,3)
    r: float = 5.0

    # interference parameter p (p=1 maximal, p=0 none)
    p: float = 1.0

    # nonradiative decay (fixed in figures)
    gamma_nr: float = 3.0

    # filter bandwidth Γ (fixed in figures)
    Gamma_filt: float = 0.1

    # excited-level splittings (scaled by gamma)
    omega21: float = 100.0
    omega32: float = 0.05

    # numerical quadrature
    Nt2: int = 320
    Ntau: int = 520


def build_liouvillian(params: Params) -> Tuple[NDArray[np.complex128], Dict[str, Dict]]:
    """
    Build 16x16 Liouvillian superoperator L acting on vec(ρ).
    Basis: |0>, |1>, |2>, |3>.
    """
    d = 4
    A = {(m, n): ketbra(d, m, n) for m in range(d) for n in range(d)}

    # S_i^+ = |i><0|, S_i^- = |0><i|
    Splus = {i: A[(i, 0)] for i in (1, 2, 3)}
    Sminus = {i: A[(0, i)] for i in (1, 2, 3)}

    # Choose ω20 as reference: ω20 = 0.
    # Then ω10 = -ω21, ω30 = +ω32 (so ω30 - ω20 = ω32).
    omega10 = -params.omega21
    omega20 = 0.0
    omega30 = params.omega32
    H0 = omega10 * A[(1, 1)] + omega20 * A[(2, 2)] + omega30 * A[(3, 3)]

    L = commutator_super(H0)

    gamma = params.gamma
    r = params.r
    # Radiative decay with broadened width (γ + r), and incoherent pump r
    for i in (1, 2, 3):
        L += lindblad_super(Sminus[i], gamma + r)  # |i> -> |0|
        L += lindblad_super(Splus[i], r)          # |0> -> |i|

    # Electron-induced interference cross terms:
    # - r_ij [ 1/2 {S_i^+S_j^- + S_j^-S_i^+, ρ} - S_j^- ρ S_i^+ - S_i^+ ρ S_j^- ]
    def interference_super(i: int, j: int, rij: float) -> NDArray[np.complex128]:
        if rij == 0.0:
            return np.zeros((d*d, d*d), dtype=np.complex128)
        Aop = Splus[i] @ Sminus[j] + Sminus[j] @ Splus[i]
        return rij * (
            -0.5 * (super_left(Aop) + super_right(Aop))
            + super_A_X_B(Sminus[j], Splus[i])
            + super_A_X_B(Splus[i], Sminus[j])
        )

    p = params.p
    rij = p * r  # since r_i=r_j=r => sqrt(r r)=r
    for i in (1, 2, 3):
        for j in (1, 2, 3):
            if i == j:
                continue
            L += interference_super(i, j, rij)

    # Nonradiative: 3->2, 3->1, 2->1 all at gamma_nr
    gnr = params.gamma_nr
    L += lindblad_super(A[(2, 3)], gnr)  # |3>->|2|
    L += lindblad_super(A[(1, 3)], gnr)  # |3>->|1|
    L += lindblad_super(A[(1, 2)], gnr)  # |2>->|1|

    op = {"A": A, "Splus": Splus, "Sminus": Sminus}
    return L, op


def gamma_ij_weights(gamma: float) -> NDArray[np.float64]:
    """
    γ_ij in the spectrum:
    γ_ii = 2γ, γ_ij = -sqrt(γγ)=-γ for i!=j (since all γ_i=γ).
    """
    G = np.zeros((3, 3), dtype=np.float64)
    for i in range(3):
        for j in range(3):
            G[i, j] = (2.0 * gamma) if (i == j) else (-gamma)
    return G

In [5]:
# ----------------------------
# Initial states used in Fig. 2(a)-(b)
# ----------------------------

def rho_pure_from_amps(amps: NDArray[np.complex128]) -> NDArray[np.complex128]:
    amps = np.asarray(amps, dtype=np.complex128).reshape((4,))
    nrm = np.sqrt(np.vdot(amps, amps).real)
    if nrm == 0:
        raise ValueError("Zero vector amplitudes.")
    psi = amps / nrm
    return np.outer(psi, psi.conj())

def fig2_initial_states() -> Dict[str, NDArray[np.complex128]]:
    """
    States as specified in the Fig. 2 caption:
    - black: |0>
    - blue: (|0>+|1>)/sqrt(2)
    - orange: equal weights, all phases 0 => (|0>+|1>+|2>+|3>)/2
    - red: equal weights, with relative phases such that |3> has phase π,
           and |1>,|2> are in phase => (|0>+|1>+|2>-|3>)/2
    """
    states = {}
    states["ground"] = ketbra(4, 0, 0)
    states["0+1"] = rho_pure_from_amps(np.array([1.0, 1.0, 0.0, 0.0], dtype=np.complex128))
    states["equal_all0"] = rho_pure_from_amps(np.array([1.0, 1.0, 1.0, 1.0], dtype=np.complex128) / 2.0)
    states["equal_pi_on3"] = rho_pure_from_amps(np.array([1.0, 1.0, 1.0, -1.0], dtype=np.complex128) / 2.0)
    return states

In [6]:
# ----------------------------
# Core spectrum computation
# ----------------------------

def compute_SCL_one_t(
    L: NDArray[np.complex128],
    op: Dict[str, Dict],
    params: Params,
    rho0: NDArray[np.complex128],
    omega: NDArray[np.float64],
    t: float
) -> NDArray[np.float64]:
    """
    Compute S_CL(omega, t) using a triangular-domain discretization:
    t2 ∈ [0,t], τ ∈ [0, t-t2].
    """
    d = 4
    A = op["A"]

    Gamma = params.Gamma_filt
    Gij = gamma_ij_weights(params.gamma)  # 3x3

    # Grids
    t2_grid = np.linspace(0.0, t, params.Nt2)
    dt2 = t2_grid[1] - t2_grid[0] if params.Nt2 > 1 else t

    # We'll use a fixed tau grid [0,t] and truncate to [0,t-t2] per t2.
    tau_grid_full = np.linspace(0.0, t, params.Ntau)
    dtau = tau_grid_full[1] - tau_grid_full[0] if params.Ntau > 1 else t

    # Precompute exp(L tau_k) for tau grid
    E_tau = [expm(L * tau) for tau in tau_grid_full]

    # Precompute rho(t2) = exp(L t2) rho0 for all t2
    rho0_vec = vec(rho0)
    rho_t2 = []
    for t2 in t2_grid:
        rho_t2.append(unvec(expm(L * t2) @ rho0_vec, d))

    # Operators A_{i0}, A_{0j}
    A_i0 = {i: A[(i, 0)] for i in (1, 2, 3)}
    A_0j = {j: A[(0, j)] for j in (1, 2, 3)}

    # Build B(t2, tau) such that:
    # S(omega,t) = Re ∫ dt2 ∫ dτ B(t2,τ) e^{-i ω τ}
    B = np.zeros((params.Nt2, params.Ntau), dtype=np.complex128)

    for it2, t2 in enumerate(t2_grid):
        pref_t2 = np.exp(-Gamma * (t - t2))
        tau_max = t - t2
        # index up to tau_max
        imax = int(np.floor(tau_max / dtau + 1e-12))
        imax = min(imax, params.Ntau - 1)

        rho2 = rho_t2[it2]

        for itau in range(imax + 1):
            tau = tau_grid_full[itau]
            Et = E_tau[itau]
            corr_sum = 0.0 + 0.0j

            # Sum over i,j = 1..3
            for j_idx, j in enumerate((1, 2, 3)):
                X0 = A_0j[j] @ rho2
                Xtau = unvec(Et @ vec(X0), d)

                for i_idx, i in enumerate((1, 2, 3)):
                    corr_ij = np.trace(A_i0[i] @ Xtau)
                    corr_sum += Gij[i_idx, j_idx] * corr_ij

            B[it2, itau] = pref_t2 * np.exp(0.5 * Gamma * tau) * corr_sum

    # Trapezoid weights for t2
    w_t2 = np.ones(params.Nt2)
    w_t2[0] *= 0.5
    w_t2[-1] *= 0.5

    # Trapezoid weights for tau (global); truncated regions are already 0
    w_tau = np.ones(params.Ntau)
    w_tau[0] *= 0.5
    w_tau[-1] *= 0.5

    # Fourier matrix: exp(-i ω τ)
    phase = np.exp(-1j * omega[:, None] * tau_grid_full[None, :])  # (Nomega, Ntau)

    # Integrate: first τ then t2
    It2 = (phase @ (B * w_tau[None, :]).T) * dtau  # (Nomega, Nt2)
    S = np.real(It2 @ (w_t2 * dt2))
    return S.astype(np.float64)


def compute_SCL_for_times(
    params: Params,
    rho0: NDArray[np.complex128],
    omega: NDArray[np.float64],
    times: List[float]
) -> Dict[float, NDArray[np.float64]]:
    L, op = build_liouvillian(params)
    out = {}
    for t in times:
        out[t] = compute_SCL_one_t(L, op, params, rho0, omega, t)
    return out

In [ ]:
# ----------------------------
# Plotting: Fig.2-style 3D waterfall
# ----------------------------

def add_filled_curve_3d(
    ax,
    x: NDArray[np.float64],
    y_const: float,
    z: NDArray[np.float64],
    alpha: float = 0.25,
    line_kwargs: Dict | None = None,
    fill: bool = True
):
    """
    Draw z(x) at fixed y=y_const in a 3D axis.
    Optionally add a filled polygon down to z=0.
    """
    if line_kwargs is None:
        line_kwargs = {}

    ax.plot(x, np.full_like(x, y_const), z, **line_kwargs)

    if fill:
        verts = [list(zip(x, z)) + [(x[-1], 0.0), (x[0], 0.0)]]
        poly = PolyCollection(verts, alpha=alpha)
        ax.add_collection3d(poly, zs=y_const, zdir="y")
        return poly
    return None


def panel_plot_initial_states(ax, with_interference: bool, outdir: str):
    """
    Panels (a) and (b):
    - parameters: gamma=1, r=5, omega21=100, omega32=0.05, Gamma=0.1, gamma_nr=3
    - times: 0.5, 1, 2
    - 4 initial states with omega-shifts for orange/red and time scaling factors.
    """
    times = [0.5, 1.0, 2.0]
    omega = np.linspace(-200.0, 200.0, 1601)

    p = 1.0 if with_interference else 0.0
    params = Params(
        gamma=1.0, r=5.0, p=p,
        gamma_nr=3.0, Gamma_filt=0.1,
        omega21=100.0, omega32=0.05,
        Nt2=320, Ntau=520
    )

    states = fig2_initial_states()

    # Compute spectra for each state
    specs = {}
    for name, rho0 in states.items():
        specs[name] = compute_SCL_for_times(params, rho0, omega, times)

    # Plot styling and shifts (as in caption)
    # orange and red shifted by +30 in omega
    omega_shift = {"ground": 0.0, "0+1": 0.0, "equal_all0": 30.0, "equal_pi_on3": 30.0}

    # time scaling factors: t=0.5 *4, t=1 *1.5, t=2 *1
    t_scale = {0.5: 4.0, 1.0: 1.5, 2.0: 1.0}

    # colors
    # ground: black (solid), 0+1: blue (filled), equal_all0: orange (filled), equal_pi_on3: red (solid)
    color_line = {
        "ground": dict(color="k", lw=1.6),
        "0+1": dict(color="#1f77b4", lw=1.2),
        "equal_all0": dict(color="#ff7f0e", lw=1.2),
        "equal_pi_on3": dict(color="r", lw=1.2),
    }
    fill_alpha = {"0+1": 0.25, "equal_all0": 0.25}

    # Draw curves at each time slice
    for t in times:
        for name in ["ground", "0+1", "equal_all0", "equal_pi_on3"]:
            z = t_scale[t] * specs[name][t]
            x = omega + omega_shift[name]

            do_fill = name in fill_alpha
            poly = add_filled_curve_3d(
                ax,
                x=x,
                y_const=t,
                z=z,
                alpha=fill_alpha.get(name, 0.0),
                line_kwargs=color_line[name],
                fill=do_fill
            )
            if poly is not None:
                poly.set_facecolor(color_line[name]["color"])

    ax.set_xlabel(r"$(\omega-\omega_{20})/\gamma$")
    ax.set_ylabel(r"time $(\gamma^{-1})$")
    ax.set_zlabel(r"$S_{\rm CL}$")

    ax.set_xlim(-200, 230)  # include shifted curves
    ax.set_ylim(0.45, 2.05)
    ax.set_yticks(times)

    ax.ticklabel_format(axis="z", style="sci", scilimits=(-2, -2))
    ax.view_init(elev=23, azim=55)

    # Save data
    os.makedirs(outdir, exist_ok=True)
    tag = "p1_initstates" if with_interference else "p0_initstates"
    np.savez_compressed(
        os.path.join(outdir, f"{tag}.npz"),
        omega=omega,
        times=np.array(times),
        params=json.dumps(asdict(params), indent=2),
        S_ground=np.stack([specs["ground"][t] for t in times], axis=0),
        S_0p1=np.stack([specs["0+1"][t] for t in times], axis=0),
        S_equal_all0=np.stack([specs["equal_all0"][t] for t in times], axis=0),
        S_equal_pi_on3=np.stack([specs["equal_pi_on3"][t] for t in times], axis=0),
    )


def panel_plot_r_sweep(ax, with_interference: bool, outdir: str):
    """
    Panels (c) and (d):
    - parameters: gamma=1, omega21=50, omega32=0.05, Gamma=0.1, gamma_nr=3
    - r ∈ {0.5, 1, 5}, initial state ground
    - times: 0.5, 1, 2
    - shifts: blue (+15), red (+45) along omega
    - time scaling: t=0.5 *15, t=1 *3
    """
    times = [0.5, 1.0, 2.0]
    omega = np.linspace(-100.0, 100.0, 1201)

    p = 1.0 if with_interference else 0.0

    r_values = [0.5, 1.0, 5.0]
    r_shift = {0.5: 0.0, 1.0: 15.0, 5.0: 45.0}
    t_scale = {0.5: 15.0, 1.0: 3.0, 2.0: 1.0}

    # colors: black (r=0.5), blue (r=1), red (r=5)
    r_style = {
        0.5: dict(color="k", lw=1.6),
        1.0: dict(color="#1f77b4", lw=1.2),
        5.0: dict(color="r", lw=1.2),
    }
    r_fill = {1.0: 0.25, 5.0: 0.25}

    rho_ground = ketbra(4, 0, 0)
    all_specs = {}

    for r in r_values:
        params = Params(
            gamma=1.0, r=r, p=p,
            gamma_nr=3.0, Gamma_filt=0.1,
            omega21=50.0, omega32=0.05,
            Nt2=320, Ntau=520
        )
        all_specs[r] = (params, compute_SCL_for_times(params, rho_ground, omega, times))

    for t in times:
        for r in r_values:
            params, spec = all_specs[r]
            z = t_scale[t] * spec[t]
            x = omega + r_shift[r]
            do_fill = r in r_fill
            poly = add_filled_curve_3d(
                ax,
                x=x,
                y_const=t,
                z=z,
                alpha=r_fill.get(r, 0.0),
                line_kwargs=r_style[r],
                fill=do_fill
            )
            if poly is not None:
                poly.set_facecolor(r_style[r]["color"])

    ax.set_xlabel(r"$(\omega-\omega_{20})/\gamma$")
    ax.set_ylabel(r"time $(\gamma^{-1})$")
    ax.set_zlabel(r"$S_{\rm CL}$")

    ax.set_xlim(-100, 150)  # include shifted curves
    ax.set_ylim(0.45, 2.05)
    ax.set_yticks(times)
    ax.ticklabel_format(axis="z", style="sci", scilimits=(-2, -2))
    ax.view_init(elev=23, azim=55)

    # Save data (one file per p)
    os.makedirs(outdir, exist_ok=True)
    tag = "p1_rsweep" if with_interference else "p0_rsweep"

    # Stack as S[r_index, t_index, omega_index]
    S_stack = np.stack(
        [np.stack([all_specs[r][1][t] for t in times], axis=0) for r in r_values],
        axis=0
    )
    np.savez_compressed(
        os.path.join(outdir, f"{tag}.npz"),
        omega=omega,
        times=np.array(times),
        r_values=np.array(r_values),
        params_list=json.dumps([asdict(all_specs[r][0]) for r in r_values], indent=2),
        S=S_stack,
    )


def save_csv_from_npz(npz_path: str):
    """
    Optional: make a flattened CSV for convenience.
    """
    if not HAVE_PANDAS:
        return
    data = np.load(npz_path, allow_pickle=True)
    omega = data["omega"]
    times = data["times"]

    # Heuristics: detect what keys exist
    records = []
    if "S" in data.files and "r_values" in data.files:
        # r-sweep file
        r_values = data["r_values"]
        S = data["S"]  # (Nr, Nt, Nomega)
        for ir, r in enumerate(r_values):
            for it, t in enumerate(times):
                for iw, w in enumerate(omega):
                    records.append((float(r), float(t), float(w), float(S[ir, it, iw])))
        df = pd.DataFrame(records, columns=["r", "t", "omega", "S"])
    else:
        # init-states file
        keys = [k for k in data.files if k.startswith("S_")]
        for k in keys:
            S = data[k]  # (Nt, Nomega)
            for it, t in enumerate(times):
                for iw, w in enumerate(omega):
                    records.append((k, float(t), float(w), float(S[it, iw])))
        df = pd.DataFrame(records, columns=["state_key", "t", "omega", "S"])

    csv_path = os.path.splitext(npz_path)[0] + ".csv"
    df.to_csv(csv_path, index=False)


def main():
    outdir = "fig2_output"
    os.makedirs(outdir, exist_ok=True)

    fig = plt.figure(figsize=(16, 4.2))

    ax1 = fig.add_subplot(1, 4, 1, projection="3d")
    panel_plot_initial_states(ax1, with_interference=True, outdir=outdir)
    ax1.set_title("(a) p = 1")

    ax2 = fig.add_subplot(1, 4, 2, projection="3d")
    panel_plot_initial_states(ax2, with_interference=False, outdir=outdir)
    ax2.set_title("(b) p = 0")

    ax3 = fig.add_subplot(1, 4, 3, projection="3d")
    panel_plot_r_sweep(ax3, with_interference=True, outdir=outdir)
    ax3.set_title("(c) p = 1")

    ax4 = fig.add_subplot(1, 4, 4, projection="3d")
    panel_plot_r_sweep(ax4, with_interference=False, outdir=outdir)
    ax4.set_title("(d) p = 0")

    plt.tight_layout()
    figpath = os.path.join(outdir, "figure2_repro.png")
    plt.savefig(figpath, dpi=220)
    print(f"Saved figure: {figpath}")

    # Also save CSV versions if pandas exists
    for fn in ["p1_initstates.npz", "p0_initstates.npz", "p1_rsweep.npz", "p0_rsweep.npz"]:
        npz_path = os.path.join(outdir, fn)
        if os.path.exists(npz_path):
            save_csv_from_npz(npz_path)

    # Combined NPZ pointer
    combined = {
        "files": ["p1_initstates.npz", "p0_initstates.npz", "p1_rsweep.npz", "p0_rsweep.npz"]
    }
    with open(os.path.join(outdir, "index.json"), "w", encoding="utf-8") as f:
        json.dump(combined, f, indent=2)
    print(f"Saved data in: {outdir}/")


if __name__ == "__main__":
    main()